In [2]:
import ultralytics 
import cv2
import PIL
from PIL import Image
import numpy as np 
import matplotlib.pyplot as plt

In [5]:
# Path to the trained YOLO weights (update this path if your weights are elsewhere)
weights_path = '/home/dang.cpm/__MY_SPACE__/Vintelligence/Github/Gesture_recognition/YOLO/runs/train/exp1/weights/epoch29.pt'

# Load the YOLO model with the specified weights
model = ultralytics.YOLO(weights_path)

In [7]:
# List of class names for the model's output classes
class_names = ['bird', 'boar', 'dog', 'dragon', 'hare', 'horse', 'monkey', 'ox', 'ram', 'rat', 'snake', 'tiger', 'zero']

# --- Real-time inference using webcam ---

# Open a connection to the default webcam (device 0)
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam.")
else:
    print("Webcam opened successfully. Press 'q' to quit.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame from webcam.")
        break

    # Convert the frame from BGR (OpenCV format) to RGB (model expects RGB)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Run YOLO inference on the frame
    # The model expects a list of images or a numpy array
    results = model(rgb_frame)

    # results[0] contains the first (and only) image's results
    boxes = results[0].boxes
    scores = results[0].scores if hasattr(results[0], 'scores') else None
    classes = results[0].cls.cpu().numpy().astype(int) if hasattr(results[0], 'cls') else []

    # Draw bounding boxes and class labels on the frame
    for i, box in enumerate(boxes):
        # Get coordinates of the bounding box
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        class_id = int(box.cls[0].cpu().numpy()) if hasattr(box, 'cls') else (classes[i] if i < len(classes) else -1)
        conf = float(box.conf[0].cpu().numpy()) if hasattr(box, 'conf') else (scores[i] if scores is not None and i < len(scores) else 0.0)
        label = class_names[class_id] if 0 <= class_id < len(class_names) else f"Class {class_id}"

        # Draw rectangle and label
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{label}: {conf:.2f}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # Display the frame with detections
    cv2.imshow('YOLO Real-Time Detection', frame)

    # Exit if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Quitting real-time detection.")
        break

# Release the webcam and close all OpenCV windows
cap.release()
# cv2.destroyAllWindows()


Error: Could not open webcam.


[ WARN:0@186.893] global cap_v4l.cpp:913 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ERROR:0@186.893] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range
